# 🛒 Shopper Spectrum: Customer Segmentation and Product Recommendations

> **Domain:** E-Commerce and Retail Analytics  
> **Problem Type:** Unsupervised ML (Clustering) + Collaborative Filtering (Recommendation System)  
> **Dataset:** Online Retail Transaction Data (2022–2023)

---

## Project Pipeline

| Step | Task |
|------|------|
| 1 | Dataset Collection & Understanding |
| 2 | Data Preprocessing |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | RFM Feature Engineering & Clustering |
| 5 | Product Recommendation System |
| 6 | Model Evaluation & Export |

---
## ⚙️ Imports & Configuration

In [ ]:
# ── Standard Library ─────────────────────────────────────────────────────────
import os
import warnings
import subprocess
import sys

warnings.filterwarnings('ignore')

# ── Data Manipulation ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_similarity

# ── Model Persistence ────────────────────────────────────────────────────────
import joblib

# ── Styling ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#3d4166',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#a0a0b0',
    'ytick.color':      '#a0a0b0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2d3154',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
    'font.size':        11,
})

PALETTE = ['#7C3AED', '#2563EB', '#059669', '#DC2626',
           '#D97706', '#DB2777', '#0891B2', '#65A30D']

print('✅ All imports successful!')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')
print(f'   Sklearn : {__import__("sklearn").__version__}')

---
## 📦 Step 1: Dataset Collection & Understanding

In [ ]:
# ── Load or generate dataset ─────────────────────────────────────────────────
DATA_PATH = 'data/online_retail.csv'

if not os.path.exists(DATA_PATH):
    print('⚠️  Dataset not found. Generating synthetic dataset ...')
    # Run the generator script
    exec(open('generate_data.py').read())
    generate_dataset(output_path=DATA_PATH)

df_raw = pd.read_csv(DATA_PATH, dtype={'CustomerID': str})
print(f'✅ Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

In [ ]:
# ── Basic Structure ──────────────────────────────────────────────────────────
print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'\nShape          : {df_raw.shape}')
print(f'Memory usage   : {df_raw.memory_usage(deep=True).sum() / 1e6:.2f} MB')
print('\nColumn data types:')
print(df_raw.dtypes.to_string())
print('\nFirst 5 rows:')
df_raw.head()

In [ ]:
# ── Missing Values Analysis ──────────────────────────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0')

print('Missing Values:')
print(missing_df.to_string())

# Visualise missing values
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(missing_df.index, missing_df['Missing %'],
              color=PALETTE[0], edgecolor='white', linewidth=0.5)
ax.set_title('Missing Value Percentage by Column', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Missing %')
ax.set_xlabel('Column')
for bar, val in zip(bars, missing_df['Missing %']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ── Duplicate Check ──────────────────────────────────────────────────────────
n_dupes = df_raw.duplicated().sum()
print(f'Duplicate rows : {n_dupes:,} ({n_dupes/len(df_raw)*100:.2f}%)')

# ── Statistical Summary ──────────────────────────────────────────────────────
print('\nNumerical Summary:')
df_raw[['Quantity', 'UnitPrice']].describe().round(2)

---
## 🧹 Step 2: Data Preprocessing

In [ ]:
print('Original shape:', df_raw.shape)
df = df_raw.copy()

# ── 1. Remove rows with missing CustomerID ────────────────────────────────────
before = len(df)
df = df.dropna(subset=['CustomerID'])
print(f'After dropping null CustomerID  : {len(df):,} rows (removed {before - len(df):,})')

# ── 2. Exclude cancelled invoices (InvoiceNo starts with 'C') ─────────────────
before = len(df)
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f'After removing cancellations    : {len(df):,} rows (removed {before - len(df):,})')

# ── 3. Remove negative or zero Quantity and UnitPrice ────────────────────────
before = len(df)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f'After removing bad qty/price    : {len(df):,} rows (removed {before - len(df):,})')

# ── 4. Remove duplicates ─────────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
print(f'After removing duplicates       : {len(df):,} rows (removed {before - len(df):,})')

# ── 5. Type conversions ───────────────────────────────────────────────────────
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID']  = df['CustomerID'].astype(str).str.strip().str.split('.').str[0]

# ── 6. Derived column: TotalAmount ───────────────────────────────────────────
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

print(f'\n✅ Clean dataset shape: {df.shape}')
print(f'   Date range        : {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
print(f'   Unique customers  : {df["CustomerID"].nunique():,}')
print(f'   Unique products   : {df["Description"].nunique():,}')
print(f'   Unique invoices   : {df["InvoiceNo"].nunique():,}')

---
## 📊 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1  Transaction Volume by Country ───────────────────────────────────────
country_sales = (
    df.groupby('Country')['TotalAmount']
    .sum()
    .sort_values(ascending=False)
    .head(12)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Transaction Analysis by Country', fontsize=15, fontweight='bold', y=1.02)

# Revenue by country
colors = [PALETTE[0] if i == 0 else PALETTE[1] for i in range(len(country_sales))]
bars = axes[0].barh(country_sales['Country'], country_sales['TotalAmount'],
                    color=colors, edgecolor='white', linewidth=0.3)
axes[0].set_title('Revenue by Country (Top 12)', fontweight='bold')
axes[0].set_xlabel('Total Revenue (£)')
axes[0].invert_yaxis()
for bar in bars:
    w = bar.get_width()
    axes[0].text(w * 1.01, bar.get_y() + bar.get_height()/2,
                 f'£{w:,.0f}', va='center', fontsize=8)

# Transaction count by country
country_tx = df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False).head(12)
axes[1].barh(country_tx.index, country_tx.values,
             color=PALETTE[2], edgecolor='white', linewidth=0.3)
axes[1].set_title('Transaction Count by Country (Top 12)', fontweight='bold')
axes[1].set_xlabel('Number of Transactions')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2  Top-Selling Products ─────────────────────────────────────────────────
top_products_qty = (
    df.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
top_products_rev = (
    df.groupby('Description')['TotalAmount']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 14))
fig.suptitle('Top-Selling Products', fontsize=15, fontweight='bold')

# By quantity
grad = [PALETTE[0]] * 5 + [PALETTE[1]] * 15
axes[0].barh(top_products_qty['Description'], top_products_qty['Quantity'],
             color=PALETTE, edgecolor='white', linewidth=0.3)
axes[0].set_title('Top 20 Products by Quantity Sold', fontweight='bold')
axes[0].set_xlabel('Total Quantity')
axes[0].invert_yaxis()

# By revenue
axes[1].barh(top_products_rev['Description'], top_products_rev['TotalAmount'],
             color=PALETTE[::-1], edgecolor='white', linewidth=0.3)
axes[1].set_title('Top 20 Products by Revenue', fontweight='bold')
axes[1].set_xlabel('Total Revenue (£)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3  Purchase Trends Over Time ────────────────────────────────────────────
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly = (
    df.groupby('YearMonth')
    .agg(Revenue=('TotalAmount', 'sum'),
         Orders=('InvoiceNo', 'nunique'),
         Customers=('CustomerID', 'nunique'))
    .reset_index()
)
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)

fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
fig.suptitle('Monthly Purchase Trends (2022–2023)', fontsize=15, fontweight='bold')

for ax, col, label, color in zip(
    axes,
    ['Revenue', 'Orders', 'Customers'],
    ['Monthly Revenue (£)', 'Monthly Orders', 'Active Customers'],
    [PALETTE[0], PALETTE[1], PALETTE[2]]
):
    ax.fill_between(monthly['YearMonth_str'], monthly[col],
                    alpha=0.3, color=color)
    ax.plot(monthly['YearMonth_str'], monthly[col],
            color=color, linewidth=2, marker='o', markersize=4)
    ax.set_ylabel(label)
    ax.grid(axis='y')
    # Mark max
    peak_idx = monthly[col].idxmax()
    ax.annotate(f'Peak: {monthly[col][peak_idx]:,.0f}',
                xy=(monthly['YearMonth_str'][peak_idx], monthly[col][peak_idx]),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=9,
                arrowprops=dict(arrowstyle='->', color='white'),
                color='white')

axes[-1].set_xlabel('Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4  Monetary Distribution per Transaction & Customer ─────────────────────
invoice_totals = df.groupby('InvoiceNo')['TotalAmount'].sum()
customer_totals = df.groupby('CustomerID')['TotalAmount'].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Monetary Distribution', fontsize=15, fontweight='bold')

# Per transaction
axes[0].hist(invoice_totals.clip(upper=invoice_totals.quantile(0.99)),
             bins=50, color=PALETTE[3], edgecolor='white', linewidth=0.3)
axes[0].set_title('Revenue per Invoice (99th percentile capped)', fontweight='bold')
axes[0].set_xlabel('Invoice Value (£)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(invoice_totals.median(), color='yellow', linestyle='--',
                label=f'Median: £{invoice_totals.median():.0f}')
axes[0].legend()

# Per customer
axes[1].hist(customer_totals.clip(upper=customer_totals.quantile(0.99)),
             bins=50, color=PALETTE[4], edgecolor='white', linewidth=0.3)
axes[1].set_title('Revenue per Customer (99th percentile capped)', fontweight='bold')
axes[1].set_xlabel('Customer Lifetime Value (£)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(customer_totals.median(), color='yellow', linestyle='--',
                label=f'Median: £{customer_totals.median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Invoice totals  – Mean: £{invoice_totals.mean():.2f}, Median: £{invoice_totals.median():.2f}, Max: £{invoice_totals.max():.2f}')
print(f'Customer totals – Mean: £{customer_totals.mean():.2f}, Median: £{customer_totals.median():.2f}, Max: £{customer_totals.max():.2f}')

---
## 🧪 Step 4: RFM Feature Engineering & Clustering

In [ ]:
# ── 4.1  Compute RFM Metrics ──────────────────────────────────────────────────
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)  # 1 day after last transaction
print(f'Snapshot date (reference): {snapshot_date.date()}')

rfm = (
    df.groupby('CustomerID')
    .agg(
        Recency  =('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
        Frequency=('InvoiceNo',   'nunique'),
        Monetary =('TotalAmount', 'sum')
    )
    .reset_index()
)

print(f'\nRFM table shape: {rfm.shape}')
print('\nRFM Summary Statistics:')
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2).to_string())

In [ ]:
# ── 4.2  RFM Distributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('RFM Score Distributions', fontsize=15, fontweight='bold')

metrics  = ['Recency', 'Frequency', 'Monetary']
colors   = [PALETTE[0], PALETTE[1], PALETTE[2]]
x_labels = ['Days since last purchase', 'Number of transactions', 'Total spend (£)']

for ax, metric, color, xlabel in zip(axes, metrics, colors, x_labels):
    data = rfm[metric].clip(upper=rfm[metric].quantile(0.99))
    ax.hist(data, bins=40, color=color, edgecolor='white', linewidth=0.3, alpha=0.9)
    ax.axvline(rfm[metric].median(), color='yellow', linestyle='--', linewidth=1.5,
               label=f'Median: {rfm[metric].median():.1f}')
    ax.axvline(rfm[metric].mean(), color='cyan', linestyle=':', linewidth=1.5,
               label=f'Mean: {rfm[metric].mean():.1f}')
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Customers')
    ax.legend(fontsize=9)
    ax.grid(axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3  Normalize RFM ────────────────────────────────────────────────────────
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])

print('✅ StandardScaler applied.')
print('Scaled statistics (should be ~0 mean, ~1 std):')
print(rfm_scaled_df.describe().round(3).to_string())

In [ ]:
# ── 4.4  Elbow Method + Silhouette Scores ─────────────────────────────────────
k_range    = range(2, 11)
inertias   = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, km.labels_))
    print(f'  K={k} → Inertia: {km.inertia_:,.1f}  |  Silhouette: {sil_scores[-1]:.4f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cluster Selection Metrics', fontsize=15, fontweight='bold')

# Elbow curve
axes[0].plot(list(k_range), inertias, marker='o', color=PALETTE[0],
             linewidth=2, markersize=8)
axes[0].fill_between(list(k_range), inertias, alpha=0.15, color=PALETTE[0])
axes[0].set_title('Elbow Curve (Inertia)', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-cluster SSE)')
axes[0].set_xticks(list(k_range))
axes[0].grid()

# Silhouette scores
best_k_idx = sil_scores.index(max(sil_scores))
bar_colors = [PALETTE[3] if i == best_k_idx else PALETTE[1] for i in range(len(sil_scores))]
axes[1].bar(list(k_range), sil_scores, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Silhouette Scores by K', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(list(k_range))
axes[1].text(list(k_range)[best_k_idx], sil_scores[best_k_idx] + 0.005,
             f'Best: K={list(k_range)[best_k_idx]}', ha='center', color='white', fontsize=10)
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

OPTIMAL_K = 4
print(f'\n🎯 Selected K = {OPTIMAL_K} clusters (aligns with business interpretation)')

In [ ]:
# ── 4.5  Train KMeans with K=4 ────────────────────────────────────────────────
kmeans = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=20, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# ── 4.6  Label Clusters ───────────────────────────────────────────────────────
cluster_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(2)
print('Cluster RFM Averages:')
print(cluster_summary.to_string())

# Assign semantic labels based on RFM profile:
# Lower Recency = more recent; Higher Frequency & Monetary = better customer
def assign_label(row):
    r, f, m = row['Recency'], row['Frequency'], row['Monetary']
    avg_r = cluster_summary['Recency'].mean()
    avg_f = cluster_summary['Frequency'].mean()
    avg_m = cluster_summary['Monetary'].mean()

    if r <= avg_r and f >= avg_f and m >= avg_m:
        return 'High-Value'
    elif r <= avg_r * 1.3 and f >= avg_f * 0.7:
        return 'Regular'
    elif r > avg_r * 1.5:
        return 'At-Risk'
    else:
        return 'Occasional'

cluster_labels_map = {
    row.name: assign_label(row) for _, row in cluster_summary.iterrows()
}
print('\nCluster → Label mapping:')
print(cluster_labels_map)

rfm['Segment'] = rfm['Cluster'].map(cluster_labels_map)

# Ensure all 4 segments exist (fallback if labelling is ambiguous)
required_segments = {'High-Value', 'Regular', 'Occasional', 'At-Risk'}
assigned_segments = set(rfm['Segment'].unique())
if not required_segments.issubset(assigned_segments):
    print('⚠️  Falling back to rank-based labelling...')
    # Rank clusters by Monetary desc then Recency asc
    ranked = cluster_summary.sort_values(['Monetary', 'Frequency'], ascending=[False, False])
    label_order = ['High-Value', 'Regular', 'Occasional', 'At-Risk']
    cluster_labels_map = {c: l for c, l in zip(ranked.index, label_order)}
    rfm['Segment'] = rfm['Cluster'].map(cluster_labels_map)

print('\n✅ Segment distribution:')
print(rfm['Segment'].value_counts().to_string())

In [ ]:
# ── 4.7  Cluster Profiles ─────────────────────────────────────────────────────
seg_colors = {
    'High-Value': '#7C3AED',
    'Regular':    '#2563EB',
    'Occasional': '#D97706',
    'At-Risk':    '#DC2626',
}

full_summary = rfm.groupby('Segment').agg(
    Customers=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(2)

print('Customer Segment Profile:')
print(full_summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Customer Segment Profiles', fontsize=15, fontweight='bold')

# Segment distribution (pie chart)
sizes = full_summary['Customers']
pie_colors = [seg_colors.get(s, '#888') for s in sizes.index]
wedges, texts, autotexts = axes[0].pie(
    sizes, labels=sizes.index, autopct='%1.1f%%',
    colors=pie_colors, startangle=140,
    textprops={'color': 'white', 'fontsize': 11},
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for a in autotexts:
    a.set_fontsize(10)
axes[0].set_title('Customer Distribution by Segment', fontweight='bold')

# Grouped bar of avg RFM
segments = full_summary.index.tolist()
x = np.arange(len(segments))
width = 0.25
r_norm = full_summary['Avg_Recency'] / full_summary['Avg_Recency'].max()
f_norm = full_summary['Avg_Frequency'] / full_summary['Avg_Frequency'].max()
m_norm = full_summary['Avg_Monetary'] / full_summary['Avg_Monetary'].max()

axes[1].bar(x - width, r_norm, width, label='Recency (norm)', color=PALETTE[0], alpha=0.85)
axes[1].bar(x,         f_norm, width, label='Frequency (norm)', color=PALETTE[1], alpha=0.85)
axes[1].bar(x + width, m_norm, width, label='Monetary (norm)', color=PALETTE[2], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(segments)
axes[1].set_ylabel('Normalised Average')
axes[1].set_title('Avg RFM Scores per Segment (Normalised)', fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.8  2D Scatter Plot (PCA) ────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
rfm_2d = pca.fit_transform(rfm_scaled)

fig, ax = plt.subplots(figsize=(12, 8))
for seg, color in seg_colors.items():
    mask = rfm['Segment'] == seg
    ax.scatter(rfm_2d[mask, 0], rfm_2d[mask, 1],
               label=seg, color=color, alpha=0.65, s=30, edgecolors='none')
ax.set_title('Customer Segments – PCA 2D Projection', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(title='Segment', fontsize=10, title_fontsize=11)
ax.grid()
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.9  3D Scatter Plot ──────────────────────────────────────────────────────
rfm_sample = rfm.sample(min(2000, len(rfm)), random_state=42)

fig_3d = px.scatter_3d(
    rfm_sample,
    x='Recency', y='Frequency', z='Monetary',
    color='Segment',
    color_discrete_map=seg_colors,
    title='Customer Segments – 3D RFM View',
    opacity=0.7,
    template='plotly_dark',
    height=650
)
fig_3d.update_traces(marker=dict(size=3))
fig_3d.show()

---
## 🤝 Step 5: Product Recommendation System

In [ ]:
# ── 5.1  Build CustomerID × Product Pivot Table ────────────────────────────────
# Aggregate at CustomerID × Description level
product_customer = (
    df.groupby(['CustomerID', 'Description'])['Quantity']
    .sum()
    .reset_index()
)

# Keep products purchased by at least 10 customers for meaningful similarity
popular_products = (
    product_customer.groupby('Description')['CustomerID']
    .nunique()
    .reset_index(rename={'CustomerID': 'n_customers'})
    .query('n_customers >= 10')['Description']
)

product_customer_filtered = product_customer[product_customer['Description'].isin(popular_products)]

# Create pivot
pivot = product_customer_filtered.pivot_table(
    index='Description',
    columns='CustomerID',
    values='Quantity',
    fill_value=0
)

print(f'Product–Customer matrix: {pivot.shape[0]} products × {pivot.shape[1]} customers')

In [ ]:
# ── 5.2  Compute Cosine Similarity ────────────────────────────────────────────
similarity_matrix = cosine_similarity(pivot)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=pivot.index,
    columns=pivot.index
)

print(f'Similarity matrix shape: {similarity_df.shape}')
print(f'Similarity value range : [{similarity_matrix.min():.4f}, {similarity_matrix.max():.4f}]')

In [ ]:
# ── 5.3  Recommendation Function ──────────────────────────────────────────────
def get_recommendations(product_name: str, top_n: int = 5) -> pd.DataFrame:
    """
    Return top_n similar products for the given product_name
    using item-based cosine similarity.
    """
    product_name = product_name.strip().upper()

    # Exact match first
    if product_name in similarity_df.index:
        scores = similarity_df[product_name].drop(product_name).sort_values(ascending=False)
    else:
        # Partial match fallback
        matches = [p for p in similarity_df.index if product_name in p]
        if not matches:
            return pd.DataFrame({'Message': [f'Product "{product_name}" not found.']})
        scores = similarity_df[matches[0]].drop(matches[0]).sort_values(ascending=False)

    return (
        scores.head(top_n)
        .reset_index()
        .rename(columns={'Description': 'Product', scores.name: 'Similarity'})
    )


# Quick test
test_product = pivot.index[0]
print(f'Testing recommendations for: "{test_product}"')
print(get_recommendations(test_product).to_string(index=False))

In [ ]:
# ── 5.4  Heatmap – Top 20 Products Similarity Matrix ──────────────────────────
top20 = (
    df.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .index.tolist()
)

# Only keep those present in the similarity matrix
top20_in_sim = [p for p in top20 if p in similarity_df.index][:20]

heatmap_data = similarity_df.loc[top20_in_sim, top20_in_sim]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cmap='RdPu',
    linewidths=0.3,
    ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Product Similarity Matrix (Top 20 Products)', fontsize=14, fontweight='bold', pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

---
## 📈 Step 6: Model Evaluation & Export

In [ ]:
# ── 6.1  Clustering Evaluation Metrics ────────────────────────────────────────
sil   = silhouette_score(rfm_scaled, rfm['Cluster'])
db    = davies_bouldin_score(rfm_scaled, rfm['Cluster'])
inert = kmeans.inertia_

print('=' * 50)
print('KMeans Clustering Evaluation')
print('=' * 50)
print(f'K (clusters)          : {OPTIMAL_K}')
print(f'Inertia               : {inert:,.2f}')
print(f'Silhouette Score      : {sil:.4f}   (higher → better, max=1)')
print(f'Davies-Bouldin Index  : {db:.4f}   (lower  → better, min=0)')

# All K scores table
eval_df = pd.DataFrame({
    'K': list(k_range),
    'Inertia': inertias,
    'Silhouette': sil_scores
})
print('\nFull evaluation table:')
print(eval_df.to_string(index=False))

In [ ]:
# ── 6.2  Final Segment Summary ────────────────────────────────────────────────
final_summary = rfm.groupby('Segment').agg(
    Count       =('CustomerID', 'count'),
    Avg_Recency =('Recency',    'mean'),
    Avg_Freq    =('Frequency',  'mean'),
    Avg_Monetary=('Monetary',   'mean'),
    Med_Monetary=('Monetary',   'median')
).round(2)

print('Final Customer Segment Summary:')
print(final_summary.to_string())

# Visualise
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Segment KPIs Dashboard', fontsize=15, fontweight='bold')

kpi_map = [
    ('Count',        'Customer Count',    PALETTE[0]),
    ('Avg_Recency',  'Avg Recency (days)',PALETTE[3]),
    ('Avg_Freq',     'Avg Frequency',     PALETTE[1]),
    ('Avg_Monetary', 'Avg Monetary (£)',  PALETTE[2]),
]

for ax, (col, title, color) in zip(axes.flat, kpi_map):
    segs = final_summary.index.tolist()
    vals = final_summary[col].values
    bars = ax.bar(segs, vals,
                  color=[seg_colors.get(s, color) for s in segs],
                  edgecolor='white', linewidth=0.5)
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{v:,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.3  Save Models ──────────────────────────────────────────────────────────
os.makedirs('models', exist_ok=True)

# KMeans model
joblib.dump(kmeans, 'models/kmeans_model.pkl')
print('✅ Saved: models/kmeans_model.pkl')

# StandardScaler
joblib.dump(scaler, 'models/scaler.pkl')
print('✅ Saved: models/scaler.pkl')

# Cluster label map
joblib.dump(cluster_labels_map, 'models/cluster_labels_map.pkl')
print('✅ Saved: models/cluster_labels_map.pkl')

# Product similarity matrix (as dataframe)
joblib.dump(similarity_df, 'models/product_similarity.pkl')
print('✅ Saved: models/product_similarity.pkl')

# Product names list
product_names = similarity_df.index.tolist()
joblib.dump(product_names, 'models/product_names.pkl')
print('✅ Saved: models/product_names.pkl')

print(f'\n🎉 All models saved. Ready for Streamlit deployment!')

In [ ]:
# ── 6.4  Save cleaned dataset with segments ────────────────────────────────────
rfm.to_csv('data/rfm_segments.csv', index=False)
print('✅ RFM segments saved → data/rfm_segments.csv')
rfm.head()